pip install xml-diff

## Initial lib testing

Imports

In [100]:
from lxml import etree
from xml_diff import compare
from xml.etree.ElementTree import ElementTree as et

Assigning docs

In [101]:
doc1 = etree.parse("../docs/localisations_1_2_0.plist")
doc2 = etree.parse("../docs/localisations_1_2_1.plist")

In [102]:
lang_keys = doc1.xpath("//key[following-sibling::*[1][self::string]]/text()")
print(lang_keys)
print(sorted(set(lang_keys)))

['version', 'en-US', 'fr', 'it', 'de', 'es', 'pt-BR', 'ru', 'tk', 'en-US', 'fr', 'it', 'de', 'es', 'pt-BR', 'ru', 'tk', 'en-US', 'es', 'pt-BR', 'fr', 'it', 'de', 'ru', 'tk']
['de', 'en-US', 'es', 'fr', 'it', 'pt-BR', 'ru', 'tk', 'version']


In [103]:
from collections import Counter
dict_counter = dict(Counter(lang_keys))
print(dict_counter)

{'version': 1, 'en-US': 3, 'fr': 3, 'it': 3, 'de': 3, 'es': 3, 'pt-BR': 3, 'ru': 3, 'tk': 3}


In [104]:
lang_keys_2 = doc2.xpath("//key[following-sibling::*[1][self::string]]/text()")
dict_counter_2 = dict(Counter(lang_keys_2))
print(dict_counter_2)


{'version': 1, 'en-US': 4, 'fr': 4, 'it': 4, 'de': 4, 'pt-BR': 4, 'ru': 4, 'tk': 4, 'jp': 4}


<hr>

Notes:
- pass a variable that names the files (maybe parse and fetch?) so that QA knows what is being handled
- ive used AI on xpath queries because last time i've used it was in 2021 and needed a quick refresher

### Compare version number

Objetive:
- to find out if the new localisation has a new version number
    - if it does -- all is good
    - if it does NOT -- must fail the test 

In [107]:
## fetch new numbered version
## gotta do a function that passes the new version so that it is not hardocded like it is atm 
loc_120_version = doc1.xpath("/plist/dict/key[.='version']/following-sibling::string[1]/text()")[0]
loc_121_version = doc2.xpath("/plist/dict/key[.='version']/following-sibling::string[1]/text()")[0]
print("Old version: ", loc_120_version)
print("New version: ", loc_121_version)

if loc_120_version != loc_121_version:
    print(f"Version number is confirmed to be different: {loc_121_version}")
else:
    print(f"Caution! Old version is being used in the new file! Change version number or verify if correct file is being inserted.")

Old version:  1.2.0
New version:  1.2.0
Caution! Old version is being used in the new file! Change version number or verify if correct file is being inserted.


### Compare the unique elements between both lists

Objective:
- if a language key is missing, that means a language is going to be left empty, having to be re-added

In [160]:
set_lang_keys = set(lang_keys)
set_lang_keys_2 = set(lang_keys_2)

unique_loc_keys_120 = sorted(set(lang_keys))
unique_loc_keys_121 = sorted(set(lang_keys_2))

unique_loc_keys_120_lenght = len(unique_loc_keys_120)
unique_loc_keys_121_lenght = len(unique_loc_keys_121)

missing_item_number = abs(unique_loc_keys_120_lenght - unique_loc_keys_121_lenght)

find_diff_locs = set_lang_keys ^ set_lang_keys_2 #symmetric difference (items in one or other but not both)
find_missing_locs = set_lang_keys_2 - set_lang_keys
#find_intersect_locs = set_lang_keys & set_lang_keys_2
##may need to add more checks for the differences
new_element_added = find_diff_locs - find_missing_locs
prev_element_miss = find_diff_locs & new_element_added
print(find_missing_locs)
print(find_diff_locs)
print(prev_element_miss)

print(f"There are {len(unique_loc_keys_120)} unique elements on localisation 1.2.0:\n {sorted(set(lang_keys))}")
print(f"There are {len(unique_loc_keys_121)} unique elements on localisation 1.2.1:\n {sorted(set(lang_keys_2))}")


## Check if length is the same
if unique_loc_keys_120_lenght == unique_loc_keys_121_lenght:
    print(f"\nSame number of elements between both lists.")
else:
    print(f"\nThere are {missing_item_number} missing translations.")

## Check if elements are the same 
if unique_loc_keys_121 == unique_loc_keys_120:
    print("\nBoth versions have the same localisation keys. No action needed.")
else:
    print(f"\nThere are missing elements! \nNew list has these new elements: different elements: {find_missing_locs} and is missing these: {prev_element_miss}.")

{'jp'}
{'es', 'jp'}
{'es'}
There are 9 unique elements on localisation 1.2.0:
 ['de', 'en-US', 'es', 'fr', 'it', 'pt-BR', 'ru', 'tk', 'version']
There are 9 unique elements on localisation 1.2.1:
 ['de', 'en-US', 'fr', 'it', 'jp', 'pt-BR', 'ru', 'tk', 'version']

Same number of elements between both lists.

There are missing elements! 
New list has these new elements: different elements: {'jp'} and is missing these: {'es'}.


### Check if all numeric keys have the same translations or if any of them are missing at least one

Objective:
- Developer could've forgotten about one localisation; imperative to have the same number and same languages throughout all keys
- Trying out BeautifulSoup

In [189]:
## Check the number of times it is repeated
lang_keys_2 = doc2.xpath("//key[following-sibling::*[1][self::string]][not(.='version')]/text()")
dict_counter_2 = dict(Counter(lang_keys_2))
print(dict_counter_2)

## Gotta also check if there are the same number of numeric keys
loc_120_version = doc1.xpath("/plist/dict/key[.='version']/following-sibling::string[1]/text()")[0]


{'en-US': 4, 'fr': 4, 'it': 4, 'de': 4, 'pt-BR': 4, 'ru': 4, 'tk': 4, 'jp': 4}


In [190]:
from bs4 import BeautifulSoup

with open("../docs/localisations_1_2_1.plist", "rb") as f:
    soup = BeautifulSoup(f, "xml") 

ver = soup.find("key", string="version").find_next_sibling("string").text
print(ver)

1.2.0


In [191]:
loc = soup.find("key", string="localisations").find_next_sibling("dict")
uuid_keys = loc.find_all("key", recursive=False)

uuid_counter = 0
for k in uuid_keys:
    print(k)
    uuid_counter+=1
print(f"There are {uuid_counter} keys.")

<key>2b827952-8d1a-4c31-9283-8753d1fc51be</key>
<key>d8225439-0a23-404c-857d-8cd37032a606</key>
<key>9bb069bc-a70c-45ce-875b-225f8ccef615</key>
<key>1981dc5a-a6b8-4fd3-be04-052522e08c86</key>
There are 4 keys.


In [275]:
## test with xpath
uuid_keys = doc2.xpath("/plist/dict/key[.='localisations']/following-sibling::dict[1]/key/text()")
print(uuid_keys)

['2b827952-8d1a-4c31-9283-8753d1fc51be', 'd8225439-0a23-404c-857d-8cd37032a606', '9bb069bc-a70c-45ce-875b-225f8ccef615', '1981dc5a-a6b8-4fd3-be04-052522e08c86']


In [276]:
list_of_vals = list()
for key, val in dict_counter_2.items():
    list_of_vals.append(val)

print(uuid_counter)
print(list_of_vals)

## Requires a true and false condition to be added, but it works
for i in list_of_vals:
    if(uuid_counter == i):
        print("Good")
    else:
        print("Stop! Missing!")

        


4
[4, 4, 4, 4, 4, 4, 4, 4]
Good
Good
Good
Good
Good
Good
Good
Good


### Gotta check if missing placeholders or empty text

Objective:
- empty strings are a usability constraint
- no placeholders breaks the game

In [233]:
## Empty detection
keys = doc2.xpath("//key[following-sibling::*[1][self::string]][not(.='version')]")
list_keys_strings = list()
for k in keys:
    pairs = (k.text, k.getnext().text)
    list_keys_strings.append(pairs)
print(list_keys_strings)

for p in list_keys_strings: #maybe save the vars in a list if many?
    if p[1] == None or "":
        print(f"Missing text in localisation: {p[0]}")

[('en-US', 'Starting in %u...'), ('fr', 'Commence dans %u...'), ('it', None), ('de', 'Start in %u...'), ('pt-BR', 'Início em %u...'), ('ru', 'До начала %...'), ('tk', 'Başlama: %u...'), ('jp', '開始まで %u...'), ('en-US', 'Watch a short video\\nand earn 15 coins.\\nYou can watch as many videos\\n as are available\\nto earn more coins.'), ('fr', "Regarde une courte vidéo\\net gagne 15 jetons.\\nTu peux voir autant de vidéos\\nqu'il y en a de disponible\\npour gagner plein de jetons."), ('it', 'Guarda un breve video\\ne ottieni 15 monete.\\nPuoi guardare tutti i video\\n mano a mano che sono disponibili\\nper ottenere altre monete.'), ('de', 'Sieh dir ein kurzes Video\\nan und gewinne 15 Münzen.\\nDu kannst dir so viele Videos ansehen,\\nwie verfügbar sind,\\num noch mehr Münzen zu holen.'), ('pt-BR', 'Assista a um vídeo curto\\ne ganhe 15 moedas.\\nVocê pode assistir a quantos vídeos\\n disponíveis quiser\\npara ganhar mais moedas.'), ('ru', 'Просмотрите короткий ролик\\nи получите 15 монет

In [263]:
## Placeholder missing or broken

string = list_keys_strings[5][1]
print(f"String: '{string}'")

parsed_string = string.split("%")
print(f"Parsed string: {parsed_string}")
print(f"First element of the parsed string: {parsed_string[1][0]}")

list_of_placeholders = ['u', '@', 'a'] #sample, will be .json

if "%" not in string:
    print("Not")
else:
    print("There is %")

if "%" not in string:
    print("Not")
else:
    if "%" in string and parsed_string[1][0] not in list_of_placeholders:
        print(f"Warning! Placeholder is missing, translation will break.\nCheck string '{string}'")
    else:
        print("Is in list")


String: 'До начала %...'
Parsed string: ['До начала ', '...']
First element of the parsed string: .
There is %
Warning! Placeholder is missing, translation will break.
Check string 'До начала %...'


In [267]:
pip install -r "../requirements.txt"

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [278]:
def get_keys(doc_vers):
        return doc_vers.xpath("//key[following-sibling::*[1][self::string]][not(.='version')]/text()")


print(get_keys(doc2))

['en-US', 'fr', 'it', 'de', 'pt-BR', 'ru', 'tk', 'jp', 'en-US', 'fr', 'it', 'de', 'pt-BR', 'ru', 'tk', 'jp', 'en-US', 'fr', 'it', 'de', 'pt-BR', 'ru', 'tk', 'jp', 'en-US', 'fr', 'it', 'de', 'ru', 'pt-BR', 'tk', 'jp']
